## Extract proteins from BindingDB

In [ ]:

# ============================================================
# BLOCK 1 — BINDINGDB TEST PROTEIN PREPARATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import hashlib
import json

ROOT = Path("/home/muhamed/repo/CTDDG/REWORKED_CODE")

BINDINGDB_ROOT = ROOT / "PREPROCESSING_OUTPUT" / "bindingdb"
EVAL_ROOT = ROOT / "EVALUATION_OUTPUT"

SPLITS = {
    "1002_103": 103,
    "1042_104": 104,
}

assert BINDINGDB_ROOT.exists(), (
    f"BindingDB folder not found: {BINDINGDB_ROOT}"
)

# ------------------------------------------------------------
# Find compressed test files automatically
# ------------------------------------------------------------

all_gz = list(BINDINGDB_ROOT.rglob("*.csv.gz"))

print("Compressed files found:", len(all_gz))

SPLIT_DATA = {}

for split_name, expected_count in SPLITS.items():

    print("\n" + "=" * 65)
    print("PROCESSING TEST FOLD:", split_name)
    print("=" * 65)

    matches = [
        p for p in all_gz
        if split_name in str(p)
        and "test" in str(p).lower()
        and p.name == "pairs_clean.csv.gz"
    ]

    if len(matches) != 1:
        print("Matching files:", matches)
        raise RuntimeError(
            f"Expected exactly one cleaned test file "
            f"for {split_name}, found {len(matches)}"
        )

    source_path = matches[0]

    print("Source:", source_path)

    # --------------------------------------------------------
    # Read protein sequences
    # --------------------------------------------------------

    df = pd.read_csv(source_path)

    print("Total protein-ligand pairs:", len(df))
    print("Columns:", list(df.columns))

    assert "protein_sequence" in df.columns, (
        "protein_sequence column not found"
    )

    sequences = (
        df["protein_sequence"]
        .dropna()
        .astype(str)
        .str.replace(r"\s+", "", regex=True)
        .str.upper()
    )

    sequences = sequences[
        sequences.str.fullmatch(r"[A-Z]+")
    ]

    unique_sequences = (
        sequences.drop_duplicates()
        .tolist()
    )

    print("Unique proteins:", len(unique_sequences))
    print("Expected original proteins:", expected_count)

    if len(unique_sequences) != expected_count:
        print(
            "WARNING: Cleaned dataset has "
            f"{len(unique_sequences)} instead of "
            f"{expected_count} original proteins."
        )

    assert unique_sequences, "No valid protein sequences!"

    # --------------------------------------------------------
    # Save fresh FASTA files and stable protein mapping
    # --------------------------------------------------------

    split_dir = EVAL_ROOT / "test_splits" / split_name
    fasta_dir = split_dir / "fasta"

    fasta_dir.mkdir(parents=True, exist_ok=True)

    records = []

    for i, sequence in enumerate(unique_sequences, 1):

        protein_id = f"protein{i}"

        fasta_path = fasta_dir / f"{protein_id}.fasta"

        fasta_path.write_text(
            f">{protein_id}\n{sequence}\n",
            encoding="utf-8"
        )

        records.append({
            "split": split_name,
            "protein_id": protein_id,
            "protein_sequence": sequence,
            "sequence_length": len(sequence),
            "sequence_sha256": hashlib.sha256(
                sequence.encode("utf-8")
            ).hexdigest(),
            "fasta_path": str(fasta_path),
        })

    manifest = pd.DataFrame(records)

    manifest_path = split_dir / "protein_manifest.csv"
    manifest.to_csv(manifest_path, index=False)

    SPLIT_DATA[split_name] = {
        "source": str(source_path),
        "num_proteins": len(unique_sequences),
        "fasta_dir": str(fasta_dir),
        "manifest": str(manifest_path),
    }

    print("\nFASTA files created:", len(records))
    print("FASTA directory:", fasta_dir)
    print("Manifest:", manifest_path)

    print(
        "Sequence lengths:",
        f"min={manifest.sequence_length.min()},",
        f"max={manifest.sequence_length.max()},",
        f"mean={manifest.sequence_length.mean():.1f}"
    )

    print("\nFirst 3 proteins:")
    print(
        manifest[
            ["protein_id", "sequence_length"]
        ].head(3).to_string(index=False)
    )

# ------------------------------------------------------------
# Save preparation summary
# ------------------------------------------------------------

summary_path = EVAL_ROOT / "test_splits" / "split_summary.json"

summary_path.write_text(
    json.dumps(SPLIT_DATA, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("BLOCK 1 COMPLETED")
print("=" * 65)

for split, details in SPLIT_DATA.items():
    print(
        f"{split}: "
        f"{details['num_proteins']} unique proteins"
    )

print("\nReady for Block 2: Fresh protein embeddings")
